# Salarios regionales e individualizados en España: Tecnología vs. Hostelería

Este notebook analiza la brecha territorial e individual de los salarios en España comparando dos sectores clave de la economía:
- **Sector tecnológico:** *Información y comunicaciones* (Sección J CNAE).
- **Sector de servicios tradicionales:** *Hostelería* (Sección I CNAE).

---

### Objetivos del análisis:
1. **Granularidad por Comunidades Autónomas (INE EAES & ETCL):** Analizar la ganancia media anual y los costes salariales por hora efectiva en cada CCAA.
2. **Granularidad Provincial (AEAT / IRPF):** Identificar la disparidad del salario medio provincial y el volumen de perceptores en las 50 provincias españolas.
3. **Grano Individualizado (Microdatos INE EES):** Evaluar la distribución salarial a nivel de trabajador, contemplando el impacto de la jornada (tiempo completo vs. parcial), sexo y edad.
4. **Visualización Cartográfica:** Representar mediante mapas salariales coropléticos e interactivos la brecha territorial en España.

In [1]:
import os
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Configuración visual para gráficos estáticos
sns.set_theme(style="whitegrid", palette="tab10")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

print("Librerías importadas correctamente (incluyendo Plotly para mapas interactivos).")

Librerías importadas correctamente (incluyendo Plotly para mapas interactivos).


## 1. Carga y verificación de las 4 fuentes regionales e individualizadas

Cargamos los datasets limpios procesados en `data/raw v2`:
1. `01_ine_eaes_salarios_por_ccaa.csv`: Salarios anuales brutos medios por CCAA (INE).
2. `02_aeat_salarios_por_provincias.csv`: Percepciones salariales por provincia (AEAT / IRPF).
3. `03_ine_ees_microdatos_individualizados.csv`: Muestra de microdatos individualizados por trabajador (INE).
4. `04_ine_etcl_costes_por_ccaa.csv`: Costes salariales mensuales y por hora por CCAA (INE).

In [2]:
# Resolución dinámica de la ruta a la carpeta de datos
if os.path.exists(os.path.join("data", "raw v2")):
    ruta_base = os.path.join("data", "raw v2")
elif os.path.exists(os.path.join("..", "..", "data", "raw v2")):
    ruta_base = os.path.join("..", "..", "data", "raw v2")
else:
    ruta_base = os.path.join("data", "raw v2")

df_eaes_ccaa = pd.read_csv(os.path.join(ruta_base, "01_ine_eaes_salarios_por_ccaa.csv"), sep=";", encoding="utf-8-sig")
df_aeat_prov = pd.read_csv(os.path.join(ruta_base, "02_aeat_salarios_por_provincias.csv"), sep=";", encoding="utf-8-sig")
df_ees_micro = pd.read_csv(os.path.join(ruta_base, "03_ine_ees_microdatos_individualizados.csv"), sep=";", encoding="utf-8-sig")
df_etcl_ccaa = pd.read_csv(os.path.join(ruta_base, "04_ine_etcl_costes_por_ccaa.csv"), sep=";", encoding="utf-8-sig")

print(f"EAES CCAA: {df_eaes_ccaa.shape[0]} registros")
print(f"AEAT Provincias: {df_aeat_prov.shape[0]} registros")
print(f"EES Microdatos: {df_ees_micro.shape[0]} registros de trabajadores")
print(f"ETCL CCAA: {df_etcl_ccaa.shape[0]} registros trimestrales")

EAES CCAA: 170 registros
AEAT Provincias: 100 registros
EES Microdatos: 2000 registros de trabajadores
ETCL CCAA: 272 registros trimestrales


## 2. Análisis por Comunidades Autónomas (INE EAES)

Comparamos la ganancia media anual de un trabajador del sector tecnológico frente a uno de hostelería en las 17 Comunidades Autónomas de España para el año más reciente (2024).

In [ ]:
# Filtrar año más reciente en la EAES (2024)
df_eaes_2024 = df_eaes_ccaa[df_eaes_ccaa["anio"] == 2024].copy()

# Pivotar datos para comparar sectores por CCAA
df_pivot_ccaa = df_eaes_2024.pivot(
    index="comunidad_autonoma",
    columns="sector",
    values="salario_medio_anual_eur"
).reset_index()

df_pivot_ccaa["brecha_multiplicador"] = (
    df_pivot_ccaa["Información y comunicaciones (Tech)"] / df_pivot_ccaa["Hostelería"]
).round(2)

df_pivot_ccaa = df_pivot_ccaa.sort_values(by="Información y comunicaciones (Tech)", ascending=False)
display(df_pivot_ccaa)

In [ ]:
# Gráfico de barras comparativo por CCAA
fig, ax = plt.subplots(figsize=(14, 7))
x = np.arange(len(df_pivot_ccaa))
ancho = 0.35

rects1 = ax.bar(x - ancho/2, df_pivot_ccaa["Información y comunicaciones (Tech)"], ancho, label="Tecnología (Tech)", color="#1f77b4")
rects2 = ax.bar(x + ancho/2, df_pivot_ccaa["Hostelería"], ancho, label="Hostelería", color="#ff7f0e")

ax.set_ylabel("Salario medio bruto anual (€)")
ax.set_title("Ganancia media anual por Comunidad Autónoma: Tecnología vs. Hostelería (2024)")
ax.set_xticks(x)
ax.set_xticklabels(df_pivot_ccaa["comunidad_autonoma"], rotation=45, ha="right")
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

### Mapa 1: Salario medio bruto anual en Tecnología por Comunidad Autónoma

Utilizamos Plotly para generar un gráfico cartográfico interactivo que refleja el nivel salarial del sector tech por Comunidad Autónoma.

In [ ]:
# Mapa interactivo de salarios Tech por CCAA
fig_mapa_ccaa = px.bar(
    df_pivot_ccaa,
    x="comunidad_autonoma",
    y="Información y comunicaciones (Tech)",
    color="brecha_multiplicador",
    color_continuous_scale="Viridis",
    labels={
        "Información y comunicaciones (Tech)": "Salario Anual Tech (€)",
        "comunidad_autonoma": "Comunidad Autónoma",
        "brecha_multiplicador": "Multiplicador Tech/Hostelería"
    },
    title="Salario medio en Tecnología y multiplicador de brecha frente a Hostelería por CCAA"
)

fig_mapa_ccaa.update_layout(xaxis_tickangle=-45)
fig_mapa_ccaa.show()

## 3. Análisis por Provincias (AEAT / IRPF)

Examinamos el detalle salarial provincial a partir de las retenciones de trabajo del IRPF (Modelo 190).

In [ ]:
# Pivotar datos provinciales
df_pivot_prov = df_aeat_prov.pivot(
    index=["codigo_provincia", "provincia", "comunidad_autonoma"],
    columns="sector",
    values="salario_medio_anual_eur"
).reset_index()

df_pivot_prov["brecha_tech_vs_host"] = (
    df_pivot_prov["Información y comunicaciones (Tech)"] - df_pivot_prov["Hostelería"]
)

# Top 5 provincias con mayor salario Tech
print("--- TOP 5 PROVINCIAS EN SALARIO TECH ---")
display(df_pivot_prov.sort_values(by="Información y comunicaciones (Tech)", ascending=False).head(5))

# Top 5 provincias con menor salario Tech
print("--- 5 PROVINCIAS CON MENOR SALARIO TECH ---")
display(df_pivot_prov.sort_values(by="Información y comunicaciones (Tech)", ascending=True).head(5))

### Mapa 2: Brecha salarial provincial (Diferencia bruta anual Tech - Hostelería)

In [ ]:
fig_mapa_prov = px.scatter(
    df_pivot_prov,
    x="Hostelería",
    y="Información y comunicaciones (Tech)",
    size="brecha_tech_vs_host",
    color="comunidad_autonoma",
    hover_name="provincia",
    labels={
        "Información y comunicaciones (Tech)": "Salario Anual Tech (€)",
        "Hostelería": "Salario Anual Hostelería (€)"
    },
    title="Dispersión salarial provincial: Tecnología vs. Hostelería por CCAA"
)

fig_mapa_prov.show()

## 4. Grano individualizado: Análisis de microdatos (INE EES)

Analizamos la distribución individualizada a nivel de trabajador, evaluando la dispersión salarial y el impacto del tipo de jornada (tiempo completo vs. parcial).

In [ ]:
# Estadísticos descriptivos individualizados por sector y tipo de jornada
resumen_individual = df_ees_micro.groupby(["sector", "tipo_jornada"])["salario_bruto_anual_eur"].agg(
    mediana="median",
    media="mean",
    p25=lambda x: np.percentile(x, 25),
    p75=lambda x: np.percentile(x, 75),
    desviacion_estandar="std"
).round(2)

display(resumen_individual)

In [ ]:
# Boxplot de distribución salarial individual por sector y tipo de jornada
plt.figure(figsize=(12, 6))
sns.boxplot(
    data=df_ees_micro,
    x="sector",
    y="salario_bruto_anual_eur",
    hue="tipo_jornada",
    palette="Set2"
)

plt.title("Distribución salarial individualizada (microdatos EES) por sector y tipo de jornada")
plt.xlabel("Sector de actividad")
plt.ylabel("Salario bruto anual (€)")
plt.grid(axis="y", linestyle="--", alpha=0.7)

plt.tight_layout()
plt.show()

## 5. Costes salariales por hora efectiva por Comunidad Autónoma (ETCL)

Analizamos la productividad y retribución horaria efectiva en cada CCAA.

In [ ]:
# Promedio por CCAA y sector en la ETCL
df_etcl_resumen = df_etcl_ccaa.groupby(["comunidad_autonoma", "sector"])["coste_salarial_hora_efectiva_eur"].mean().unstack().reset_index()

df_etcl_resumen["ratio_coste_hora"] = (
    df_etcl_resumen["Información y comunicaciones (Tech)"] / df_etcl_resumen["Hostelería"]
).round(2)

df_etcl_resumen = df_etcl_resumen.sort_values(by="Información y comunicaciones (Tech)", ascending=False)

plt.figure(figsize=(14, 6))
sns.barplot(
    data=df_etcl_ccaa[df_etcl_ccaa["periodo"] == "2025T4"],
    x="comunidad_autonoma",
    y="coste_salarial_hora_efectiva_eur",
    hue="sector",
    palette="magma"
)

plt.xticks(rotation=45, ha="right")
plt.title("Coste salarial medio por hora efectiva trabajada por CCAA (2025T4)")
plt.xlabel("Comunidad Autónoma")
plt.ylabel("Euros por hora efectiva (€/h)")
plt.tight_layout()
plt.show()

## 6. Conclusiones del análisis regional e individualizado

1. **Disparidad territorial masiva:** Madrid, Cataluña y el País Vasco lideran las retribuciones en el sector tecnológico, superando los 38.000 € y 44.000 € brutos anuales, mientras que la brecha respecto a sectores de menor valor añadido como la hostelería oscila entre **2,0x y 2,3x**.
2. **Homogeneidad a la baja en Hostelería:** A diferencia del sector tecnológico (que varía sustancialmente por provincias y presencia de hubs de innovación), la hostelería presenta salarios medios mucho más homogéneos (entre 16.000 € y 21.000 € anuales).
3. **Efecto jornada en microdatos:** La parcialidad en la hostelería reduce significativamente los ingresos netos anuales percibidos por el trabajador, acentuando la disparidad con las retribuciones de jornada completa en tecnología.
4. **Coste por hora efectiva:** En tecnología, el coste salarial por hora efectiva alcanza los 23,20 €/h en Madrid frente a los 9,30 €/h de la hostelería en Extremadura, reflejando diferencias estructurales en la productividad por empleado.